In [1]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,LabelEncoder,StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from xgboost import XGBClassifier
import warnings
warnings.simplefilter(action = 'ignore')

In [2]:
X_train = pd.read_csv('https://huggingface.co/datasets/NHANGIOI/Learning-Training/resolve/main/spaceship-titanic/train.csv',index_col = 'PassengerId')
y_train = X_train['Transported'].map({
    True : 1,
    False : 0
})
X_train.drop(columns = 'Transported',inplace = True)

In [3]:
X_train.drop(columns = 'VIP',inplace = True)
X_train.drop(columns = 'Name',inplace = True)
X_train.head()

,HomePlanet,CryoSleep,Cabin,Destination,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
PassengerId,,,,,,,,,,
0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,0.0,0.0,0.0,0.0,0.0
0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,109.0,9.0,25.0,549.0,44.0
0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,43.0,3576.0,0.0,6715.0,49.0
0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,0.0,1283.0,371.0,3329.0,193.0
0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,303.0,70.0,151.0,565.0,2.0


In [4]:
for col in X_train.columns:
    print(X_train[col].dtype)

object
object
object
object
float64
float64
float64
float64
float64
float64


In [5]:
num_cols = [col for col in X_train.columns if X_train[col].dtype == 'float']
cat_cols = [col for col in X_train.columns if col not in num_cols]
print(num_cols)
print(cat_cols)

['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
['HomePlanet', 'CryoSleep', 'Cabin', 'Destination']


In [6]:
for col in cat_cols:
    res = pd.unique(X_train[col])
    print(f"{col} : {res.shape[0]}")

HomePlanet : 4
CryoSleep : 3
Cabin : 6561
Destination : 4


In [7]:
Oh_cat_cols = ['HomePlanet','Destination']
Or_cat_cols = ['Cabin','CryoSleep']

In [8]:
preprocessing = ColumnTransformer(transformers = [
    ('num_cols',StandardScaler(),num_cols),
    ('Oh_cat_cols',Pipeline(steps = [
        ('fill_missing',SimpleImputer(strategy = 'most_frequent')),
        ('encode',OneHotEncoder(handle_unknown = 'ignore'))
    ]),Oh_cat_cols),
    ('Or_cat_cols',Pipeline(steps = [
        ('fill_missing',SimpleImputer(strategy = 'most_frequent')),
        ('encode',OrdinalEncoder(handle_unknown = 'use_encoded_value',unknown_value = -1))
    ]),Or_cat_cols)
])

In [9]:
xgb_model = Pipeline(steps = [
    ('preproccess',preprocessing),
    ('model',XGBClassifier(
        n_estimators = 500,
        max_depth = 8,
        subsample = 0.8,
        colsample_bytree = 0.8,
        learning_rate = 0.01,
        random_state = 42,
        n_jobs = -1
    ))
])

In [10]:
xgb_model.fit(X_train,y_train)

Pipeline(steps=[('preproccess',
                 ColumnTransformer(transformers=[('num_cols', StandardScaler(),
                                                  ['Age', 'RoomService',
                                                   'FoodCourt', 'ShoppingMall',
                                                   'Spa', 'VRDeck']),
                                                 ('Oh_cat_cols',
                                                  Pipeline(steps=[('fill_missing',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encode',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['HomePlanet',
                                                   'Destination']),
                                                 ('Or_cat_cols',
                                                  Pipeline...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.01,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=8, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=500, n_jobs=-1,
                               num_parallel_tree=None, ...))])

In [27]:
X_test = pd.read_csv('https://huggingface.co/datasets/NHANGIOI/Learning-Training/resolve/main/spaceship-titanic/test.csv',index_col = 'PassengerId')
X_test.drop(columns = 'VIP',inplace = True)
X_test.drop(columns = 'Name',inplace = True)
preds = xgb_model.predict(X_test)

In [29]:
preds = pd.Series(preds).map({
    1 : True,
    0 : False
})

submission = pd.DataFrame({
    'PassengerId' : X_test.index,
    'Transported' : preds
})
submission.to_csv('submission.csv',index = False)